In [1]:
#!/usr/bin/env python3
"""
下载DementiaBank所有0wav目录
包括Control和Dementia组
"""

import urllib.request
import urllib.parse
import http.cookiejar
from bs4 import BeautifulSoup
import time
from pathlib import Path
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s'
)
logger = logging.getLogger(__name__)


class WavDownloaderAll:
    """完整的WAV文件下载器"""

    def __init__(self, cookie_file, output_dir='../Pitt_Data/Pitt_Corpus/Pitt_audio_wav'):
        self.base_url = 'https://media.talkbank.org/dementia/English/Pitt/'
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # 加载cookies
        self.cookie_jar = http.cookiejar.MozillaCookieJar(cookie_file)
        try:
            self.cookie_jar.load(ignore_discard=True, ignore_expires=True)
            logger.info(f"✓ 加载了 {len(self.cookie_jar)} 个cookies")
        except Exception as e:
            logger.error(f"✗ 加载cookie失败: {e}")
            raise

        # 创建opener
        self.opener = urllib.request.build_opener(
            urllib.request.HTTPCookieProcessor(self.cookie_jar)
        )

        self.opener.addheaders = [
            ('User-Agent', 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'),
            ('Accept', '*/*'),
            ('Accept-Language', 'en-US,en;q=0.9'),
            ('Referer', 'https://sla.talkbank.org/TBB/ca'),
            ('Connection', 'keep-alive'),
        ]

        urllib.request.install_opener(self.opener)

        self.stats = {
            'total': 0,
            'downloaded': 0,
            'skipped': 0,
            'failed': 0
        }

    def download_file(self, url, save_path, retry=3):
        """下载单个文件"""

        # 检查已存在的文件
        if save_path.exists():
            file_size = save_path.stat().st_size
            if file_size > 100000:
                logger.info(f"  ✓ 跳过（已存在）: {save_path.name} ({file_size/1024/1024:.1f}MB)")
                self.stats['skipped'] += 1
                return True
            else:
                logger.warning(f"  删除不完整文件: {save_path.name} ({file_size}B)")
                save_path.unlink()

        # 添加?f=save参数
        download_url = url + '?f=save'

        for attempt in range(retry):
            try:
                if attempt > 0:
                    logger.info(f"    重试 {attempt+1}/{retry}")

                req = urllib.request.Request(download_url)
                start_time = time.time()

                with urllib.request.urlopen(req, timeout=120) as response:
                    if response.status != 200:
                        logger.warning(f"    警告: 状态码 {response.status}")
                        if attempt < retry - 1:
                            time.sleep(2)
                            continue
                    content = response.read()
                    elapsed = time.time() - start_time

                # 验证文件
                if len(content) < 1000:
                    logger.error(f"    ✗ 文件太小: {len(content)}B")
                    if attempt < retry - 1:
                        time.sleep(2)
                        continue
                    else:
                        self.stats['failed'] += 1
                        return False

                if content.startswith(b'<!DOCTYPE') or content.startswith(b'<html'):
                    logger.error(f"    ✗ 下载到HTML页面")
                    self.stats['failed'] += 1
                    return False

                is_audio = (
                    content.startswith(b'ID3') or
                    content[0:2] in [b'\xff\xfb', b'\xff\xf3', b'\xff\xf2'] or
                    content.startswith(b'RIFF')
                )

                if not is_audio:
                    logger.error(f"    ✗ 不是音频文件")
                    self.stats['failed'] += 1
                    return False

                # 保存文件
                with open(save_path, 'wb') as f:
                    f.write(content)

                speed = len(content) / elapsed / 1024 if elapsed > 0 else 0
                logger.info(f"  ✓ 成功: {save_path.name} ({len(content)/1024/1024:.1f}MB, {speed:.0f}KB/s)")
                self.stats['downloaded'] += 1
                return True

            except Exception as e:
                logger.error(f"    ✗ 下载失败: {e}")
                if attempt < retry - 1:
                    time.sleep(2)
                else:
                    self.stats['failed'] += 1
                    return False

        return False

    def get_audio_links(self, url):
        """获取音频文件链接"""
        try:
            req = urllib.request.Request(url)
            with urllib.request.urlopen(req, timeout=30) as response:
                html = response.read().decode('utf-8')

            soup = BeautifulSoup(html, 'html.parser')
            audio_links = []

            for link in soup.find_all('a', href=True):
                href = link['href']
                if (href.lower().endswith('.wav') and
                    '?' not in href and
                    not href.startswith('/')):
                    full_url = urllib.parse.urljoin(url, href)
                    audio_links.append(full_url)

            audio_links = list(set(audio_links))
            audio_links.sort()
            return audio_links

        except Exception as e:
            logger.error(f"获取链接失败: {e}")
            return []

    def check_0wav_exists(self, category):
        """检查0wav目录是否存在"""
        url = f"{self.base_url}{category}/cookie/0wav/"
        try:
            req = urllib.request.Request(url)
            response = urllib.request.urlopen(req, timeout=10)
            return response.status == 200
        except:
            return False

    def download_category_0wav(self, category):
        """下载某个类别的0wav目录"""

        url = f"{self.base_url}{category}/cookie/0wav/"

        logger.info(f"\n{'='*80}")
        logger.info(f"类别: {category}")
        logger.info(f"{'='*80}")

        # 检查目录是否存在
        if not self.check_0wav_exists(category):
            logger.warning(f"{category} 组没有0wav目录，跳过")
            return

        logger.info(f"扫描: {url}")

        # 创建子目录
        category_dir = self.output_dir / category
        category_dir.mkdir(parents=True, exist_ok=True)

        # 获取音频链接
        audio_links = self.get_audio_links(url)

        if not audio_links:
            logger.warning(f"没有找到WAV文件")
            return

        self.stats['total'] += len(audio_links)
        logger.info(f"发现 {len(audio_links)} 个WAV文件")
        logger.info(f"{'='*80}\n")

        # 下载每个文件
        for i, audio_url in enumerate(audio_links, 1):
            filename = Path(urllib.parse.urlparse(audio_url).path).name
            save_path = category_dir / filename

            logger.info(f"[{i}/{len(audio_links)}] {filename}")
            self.download_file(audio_url, save_path)
            time.sleep(0.3)

    def download_all(self):
        """下载所有0wav目录"""

        logger.info(f"\n{'='*80}")
        logger.info(f"DementiaBank WAV文件下载（0wav目录）")
        logger.info(f"{'='*80}")
        logger.info(f"输出目录: {self.output_dir.absolute()}")
        logger.info(f"{'='*80}")

        start_time = time.time()

        # 下载Control组
        self.download_category_0wav('Control')

        # 下载Dementia组
        self.download_category_0wav('Dementia')

        elapsed = time.time() - start_time

        # 总结
        logger.info(f"\n\n{'='*80}")
        logger.info("下载完成！")
        logger.info(f"{'='*80}")
        logger.info(f"总数:   {self.stats['total']}")
        logger.info(f"成功:   {self.stats['downloaded']}")
        logger.info(f"跳过:   {self.stats['skipped']}")
        logger.info(f"失败:   {self.stats['failed']}")
        logger.info(f"耗时:   {elapsed/60:.1f} 分钟")
        logger.info(f"输出:   {self.output_dir.absolute()}")
        logger.info(f"{'='*80}")


def main():
    import sys

    cookie_file = 'cookies.txt'

    if not Path(cookie_file).exists():
        logger.error(f"\n错误: Cookie文件不存在: {cookie_file}")
        sys.exit(1)

    try:
        downloader = WavDownloaderAll(
            cookie_file=cookie_file,
            output_dir='../Data/Pitt_Corpus/Pitt_audio_wav'
        )

        downloader.download_all()

    except KeyboardInterrupt:
        logger.info("\n\n用户中断下载")
    except Exception as e:
        logger.error(f"\n错误: {e}")
        import traceback
        traceback.print_exc()


if __name__ == '__main__':
    main()

2025-11-11 19:18:06,155 - ✓ 加载了 3127 个cookies
2025-11-11 19:18:06,162 - 
2025-11-11 19:18:06,162 - DementiaBank WAV文件下载（0wav目录）
2025-11-11 19:18:06,163 - ================================================================================
2025-11-11 19:18:06,163 - 输出目录: /Users/sleepwalker/Library/Mobile Documents/com~apple~CloudDocs/Code-In-iCloud/Adaptation_is_all_you_need/extract_patient_speech/Download/../Data/Pitt_Corpus/Pitt_audio_wav
2025-11-11 19:18:06,163 - ================================================================================
2025-11-11 19:18:06,163 - 
2025-11-11 19:18:06,163 - 类别: Control
2025-11-11 19:18:06,163 - ================================================================================
2025-11-11 19:18:06,449 - 扫描: https://media.talkbank.org/dementia/English/Pitt/Control/cookie/0wav/
2025-11-11 19:18:06,598 - 没有找到WAV文件
2025-11-11 19:18:06,599 - 
2025-11-11 19:18:06,599 - 类别: Dementia
2025-11-11 19:18:06,600 - =====================================================